# Run the processing pipeline

Builds the episode table (Stages 1–3) and runs download, transcription, ad removal and spontaneity check (Stages 4–7).

Run the cells from top to bottom. Transcription costs API credit, so do the smoke test before the full run.
The episode table and the processed table are saved in `data/`, so a later session can continue where the last one stopped.

## 1. Setup

In [ ]:
import sys
import logging
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd().parent
for folder in [PROJECT_DIR, PROJECT_DIR / "preprocessing"]:    # project root holds config.py
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import config
from fetch_pipeline import FetchPipeline
from filter_index import PodcastDatabase
from extract_episodes import EpisodeFetcher
from download import EpisodeDownloader
from transcribe import EpisodeTranscriber
from remove_ads import AdRemover
from classify_spontaneity import SpontaneityClassifier
from processing_pipeline import ProcessingPipeline

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

### Parameters

All settings of a run are in `config.py` in the project folder (years, groups and filters, `K`, `N_PODCASTS`, ad rule, models, paths).
Change them there, not here, and restart the kernel afterwards. This cell only shows the values of this run.

In [ ]:
print(f"Years: {config.YEARS[0]}-{config.YEARS[-1]}, screening on {config.SCREENING_YEARS}")
print(f"Groups: {list(config.GROUPS)}")
print(f"N_PODCASTS = {config.N_PODCASTS}, K = {config.K}, K_TEST = {config.K_TEST}, BUFFER = {config.BUFFER}")
print(f"Excerpt: second {config.START_SEC} to {config.START_SEC + config.DURATION_SEC}, at least {config.MIN_WORDS_PER_EPISODE} words after ad removal")

## 2. Build the episode table

Reads the RSS feeds of all matching podcasts, which takes a while. The table is saved and loaded from the file in later sessions.
Delete `data/episodes.pkl` to build it again, for example after changing `GROUPS`, `K`, `BUFFER` or `YEARS` in `config.py`.

In [ ]:
if config.EPISODES_PATH.exists():
    episodes = pd.read_pickle(config.EPISODES_PATH)
else:
    tables = []
    for group, filters in config.GROUPS.items():
        pod_db = PodcastDatabase(str(config.DATABASE_PATH), config.DATABASE_TABLE)
        fetch_pipeline = FetchPipeline(pod_db, EpisodeFetcher(), group, filters=filters)
        tables.append(fetch_pipeline.build_podcast_df(k=config.K, buffer=config.BUFFER))
    episodes = pd.concat(tables, ignore_index=True)
    episodes.to_pickle(config.EPISODES_PATH)

print(episodes.groupby("group")["feed_url"].nunique().rename("candidate podcasts per group (pool, the pipeline takes the first N_PODCASTS that pass)"))
episodes.head()

## 3. Create the pipeline

In [ ]:
pipeline = ProcessingPipeline(
    ep_downloader=EpisodeDownloader(config.DOWNLOAD_DIR),
    ep_transcriber=EpisodeTranscriber(config.TRANSCRIPTION_DIR),
    ad_remover=AdRemover(config.AD_REMOVED_TRANSCRIPTION_DIR),
    classifier=SpontaneityClassifier(),
)

## 4. Smoke test

One podcast per group with one episode per year. Check the result and the cost before the full run.
Files written here are reused by the full run.

In [ ]:
smoke = pipeline.process(episodes.copy(), k=1, n_podcasts=1, k_test=config.K_TEST)
smoke[smoke["episode_status"] != "pending"][["podcast_title", "podcast_status", "podcast_fail_reason", "episode_year", "episode_status", "episode_fail_reason",
                                             "n_words", "n_speakers", "score", "scripted"]]

## 5. Full run

Continues from `data/processed.pkl` if it exists, so nothing is processed twice.
If the run is interrupted, run this cell again: finished downloads, transcripts and ad files are reused from disk.

In [ ]:
start = pd.read_pickle(config.PROCESSED_PATH) if config.PROCESSED_PATH.exists() else episodes.copy()

processed = pipeline.process(start, k=config.K, n_podcasts=config.N_PODCASTS, k_test=config.K_TEST)
processed.to_pickle(config.PROCESSED_PATH)

## 6. Results

Podcasts per group and status. `pending` podcasts were not needed.

In [ ]:
podcasts = processed.drop_duplicates("feed_url")
podcasts.groupby(["group", "podcast_status"]).size().unstack(fill_value=0)

Processed episodes per podcast and year. Every cell of a passed podcast should hold `K`.

In [ ]:
done = processed[(processed["podcast_status"] == "passed") & (processed["episode_status"] == "processed")]
done.groupby(["group", "podcast_title", "episode_year"]).size().unstack(fill_value=0)

Why podcasts and episodes failed.

In [ ]:
print(podcasts[podcasts["podcast_status"] == "failed"]["podcast_fail_reason"].value_counts(), "\n")

failed = processed[processed["episode_status"] == "failed"]
print(failed["fail_stage"].value_counts())
failed[["podcast_title", "episode_year", "fail_stage", "episode_fail_reason"]]

Spontaneity score and transcription quality per group and year (processed episodes of passed podcasts).

In [ ]:
done.astype({"score": float, "mean_confidence": float, "n_words": float}).groupby(["group", "episode_year"])[["score", "mean_confidence", "n_words"]].mean().round(2)